# 02 · Calendario de la tienda 44

Construye una tabla con **una fila por día con operación** y todas las variables de calendario que se usarán como controles, más las transacciones diarias de la tienda (solo para el modelo de tráfico).

| | |
|---|---|
| **Entrada** | `../data/holidays_events.csv`, `../data/transactions.csv`, `../data/stores.csv`, `../data/procesados/panel_t44_sku_dia.parquet` (solo las fechas) |
| **Salida** | `../data/procesados/calendario_t44.csv` |
| **Decisiones que implementa** | D‑09 (terremoto), D‑10 (feriados y eventos), D‑12 (`transactions`), D‑13 (días de pago), D‑16 (efectos fijos de calendario) |

**Qué corrige.** El EDA anterior eliminaba filas repetidas por fecha en `holidays_events.csv` **antes** de filtrar por ciudad, y así perdía 7 fechas relevantes para la tienda 44, entre ellas el Viernes Santo 2017 y el Día de la Madre 2016 (P‑03). Aquí se filtra primero lo que aplica a la tienda (nacional + Pichincha + Quito) y luego se combinan las filas de una misma fecha con un "o" lógico, sin descartar ninguna.

In [1]:
# ------------------------------------------------------------------------------------------------------------
# Este notebook construye el calendario diario de la tienda 44 (feriados, eventos, días de pago, terremoto y
# transacciones).
# ------------------------------------------------------------------------------------------------------------
# Implementación: Ramiro Saltos
# Fecha de versión: 2026-10-15
# Revisado: 2026-10-15
# ------------------------------------------------------------------------------------------------------------

# Librerías
# ------------------------------------------------------------------------------------------------------------
import os

import numpy as np
import pandas as pd

In [2]:
# Parámetros
# ------------------------------------------------------------------------------------------------------------
tienda = 44                                          # D-04
ventana_terremoto = ("2016-04-16", "2016-05-31")     # D-09: días excluidos de la estimación
dias_post_pago = 2                                   # D-13: días siguientes al día de pago

# Rutas
# ------------------------------------------------------------------------------------------------------------
ruta_datos = os.path.join("..", "data")
ruta_procesados = os.path.join(ruta_datos, "procesados")
archivo_panel = os.path.join(ruta_procesados, "panel_t44_sku_dia.parquet")
archivo_salida = os.path.join(ruta_procesados, "calendario_t44.csv")

In [3]:
# Leer los días con operación (del panel) y las tablas auxiliares con tipos declarados
dias = pd.DataFrame({"date": np.sort(pd.read_parquet(archivo_panel, columns=["date"])["date"].unique())})

feriados = pd.read_csv(
    os.path.join(ruta_datos, "holidays_events.csv"),
    parse_dates=["date"],
    dtype={"type": "str", "locale": "str", "locale_name": "str", "description": "str", "transferred": "bool"},
)
transacciones = pd.read_csv(os.path.join(ruta_datos, "transactions.csv"), parse_dates=["date"])
tiendas = pd.read_csv(os.path.join(ruta_datos, "stores.csv"))

# Obtener la ciudad y la provincia de la tienda
ciudad, provincia = tiendas.loc[tiendas["store_nbr"] == tienda, ["city", "state"]].iloc[0]
print(f"Tienda {tienda}: {ciudad}, {provincia} | días con operación: {len(dias)}")

Tienda 44: Quito, Pichincha | días con operación: 590


## 1. Feriados y eventos (D‑10)

Semántica de `holidays_events.csv`: un `Holiday` con `transferred = True` **no** se celebró ese día; `Transfer` es el día al que se trasladó; `Bridge` es un día libre extra; `Work Day` es un día normalmente libre que se trabaja; `Additional` son días añadidos a un feriado; `Event` son eventos. Se crean seis indicadores separados porque sus efectos sobre la demanda difieren:

| Indicador | Qué marca |
|---|---|
| `dia_libre` | `Holiday` no trasladado, `Transfer` o `Bridge` |
| `adicional` | `Additional` (p. ej., días previos a Navidad) |
| `dia_laboral_recuperado` | `Work Day` |
| `terremoto` | eventos del terremoto de 2016 |
| `evento_comercial` | Black Friday y Cyber Monday (son promociones en sí mismas) |
| `evento_otro` | el resto de los eventos (p. ej., Día de la Madre) |

In [4]:
def construir_calendario_feriados(
    feriados: pd.DataFrame,
    dias: pd.DataFrame,
    ciudad: str,
    provincia: str,
) -> pd.DataFrame:
    """
    Esta función construye los indicadores de feriados y eventos que aplican a una tienda, sin descartar filas
    cuando una fecha tiene varios registros.

    :param feriados:    el contenido de holidays_events.csv.
    :param dias:        un DataFrame con la columna date de los días con operación de la tienda.
    :param ciudad:      la ciudad de la tienda (feriados locales).
    :param provincia:   la provincia de la tienda (feriados regionales).
    :return:            un DataFrame con una fila por día con operación y seis indicadores booleanos.
    """

    # Conservar solo los registros que aplican a la tienda: nacionales, de su provincia y de su ciudad
    aplica = (
        (feriados["locale"] == "National")
        | ((feriados["locale"] == "Regional") & (feriados["locale_name"] == provincia))
        | ((feriados["locale"] == "Local") & (feriados["locale_name"] == ciudad))
    )
    f = feriados.loc[aplica].copy()

    # Crear los indicadores a partir del tipo y la descripción
    descripcion = f["description"].str.lower()
    f["dia_libre"] = ((f["type"] == "Holiday") & ~f["transferred"]) | f["type"].isin(["Transfer", "Bridge"])
    f["adicional"] = f["type"] == "Additional"
    f["dia_laboral_recuperado"] = f["type"] == "Work Day"
    f["terremoto"] = descripcion.str.contains("terremoto")
    f["evento_comercial"] = descripcion.str.contains("black friday|cyber monday")
    f["evento_otro"] = (f["type"] == "Event") & ~f["terremoto"] & ~f["evento_comercial"]

    # Combinar las filas de una misma fecha con un "o" lógico y llevar a los días con operación
    indicadores = ["dia_libre", "adicional", "dia_laboral_recuperado", "terremoto", "evento_comercial", "evento_otro"]
    por_fecha = f.groupby("date")[indicadores].max()
    calendario = dias.merge(por_fecha, left_on="date", right_index=True, how="left")
    calendario[indicadores] = calendario[indicadores].astype("boolean").fillna(False).astype(bool)

    return calendario


calendario = construir_calendario_feriados(feriados, dias, ciudad, provincia)

# Verificar fechas que la versión anterior perdía (P-03)
fila = calendario.set_index("date")
assert fila.loc["2017-04-14", "dia_libre"], "el Viernes Santo 2017 debe ser día libre"
assert fila.loc["2016-05-08", "evento_otro"], "el Día de la Madre 2016 debe estar marcado"
assert fila.loc["2016-05-01", "dia_libre"] and fila.loc["2016-05-01", "terremoto"], "el 1-5-2016 tiene feriado y evento"
assert fila.loc["2016-11-12", "dia_laboral_recuperado"], "el 12-11-2016 es día laboral recuperado"

print(calendario.drop(columns="date").sum().rename("días marcados").to_string())

dia_libre                 18
adicional                  9
dia_laboral_recuperado     1
terremoto                 31
evento_comercial           2
evento_otro                2


## 2. Días de pago (D‑13) y ventana del terremoto (D‑09)

Según la documentación de Kaggle, los salarios del sector público se pagan los días 15 y el último día de cada mes. `post_pago` marca los dos días calendario siguientes.

La **ventana del terremoto** (16‑04‑2016 a 31‑05‑2016, 46 días) se excluye de la estimación en el análisis principal (D‑09). Aquí solo se marca; la exclusión se hace al armar la base de cada modelo. El indicador `terremoto` del punto anterior queda para el análisis de robustez.

In [5]:
def es_dia_de_pago(fechas: pd.Series) -> pd.Series:
    """
    Esta función indica si cada fecha es día de pago del sector público (día 15 o último día del mes).

    :param fechas:  una serie de fechas.
    :return:        una serie booleana del mismo largo.
    """

    return (fechas.dt.day == 15) | fechas.dt.is_month_end


# Marcar el día de pago y los días calendario siguientes
calendario["dia_pago"] = es_dia_de_pago(calendario["date"])
calendario["post_pago"] = False
for k in range(1, dias_post_pago + 1):
    calendario["post_pago"] |= es_dia_de_pago(calendario["date"] - pd.Timedelta(days=k))

# Marcar la ventana del terremoto
inicio_terremoto, fin_terremoto = pd.Timestamp(ventana_terremoto[0]), pd.Timestamp(ventana_terremoto[1])
calendario["ventana_terremoto"] = calendario["date"].between(inicio_terremoto, fin_terremoto)

print(f"Días de pago:            {calendario['dia_pago'].sum()}")
print(f"Días post pago:          {calendario['post_pago'].sum()}")
print(f"Días en la ventana del terremoto: {calendario['ventana_terremoto'].sum()} "
      f"(quedan {(~calendario['ventana_terremoto']).sum()} para estimar)")

Días de pago:            39
Días post pago:          76
Días en la ventana del terremoto: 46 (quedan 544 para estimar)


## 3. Variables para los efectos fijos de calendario (D‑16) y transacciones (D‑12)

`dow` es el día de la semana (0 = lunes, 6 = domingo); `semana` es la semana ISO; `mes` es el año‑mes. `transactions` es el número de boletas de la tienda en el día: **no se usa como control** en las ecuaciones de ventas, porque es el canal por el que una promoción atraería tráfico (D‑12); es la variable dependiente del modelo de tráfico.

In [6]:
# Crear las variables de calendario
iso = calendario["date"].dt.isocalendar()
calendario["dow"] = calendario["date"].dt.dayofweek.astype("int8")
calendario["semana"] = iso["year"].astype(str) + "-S" + iso["week"].astype(str).str.zfill(2)
calendario["mes"] = calendario["date"].dt.strftime("%Y-%m")

# Agregar las transacciones de la tienda
trans_tienda = transacciones.loc[transacciones["store_nbr"] == tienda, ["date", "transactions"]]
calendario = calendario.merge(trans_tienda, on="date", how="left")

# Verificar la granularidad y los faltantes conocidos de transactions (2016-01-03 y 2016-01-04)
assert len(calendario) == len(dias), "el calendario debe tener una fila por día con operación"
assert not calendario["date"].duplicated().any(), "hay fechas repetidas en el calendario"
faltantes = calendario.loc[calendario["transactions"].isna(), "date"].dt.date.tolist()
print(f"Días sin dato de transactions: {faltantes}")

Días sin dato de transactions: [datetime.date(2016, 1, 3), datetime.date(2016, 1, 4)]


## 4. Exportación

In [7]:
# Guardar el calendario en CSV: es una tabla pequeña que se puede abrir en Excel (A-07)
calendario.to_csv(archivo_salida, index=False)
print(f"Guardado: {archivo_salida}  ({len(calendario)} días, {calendario.shape[1]} columnas)")

print(f"\n>>>> Proceso finalizado <<<<")

Guardado: ..\data\procesados\calendario_t44.csv  (590 días, 14 columnas)

>>>> Proceso finalizado <<<<
